#### Static

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import dgl
import dgl.nn as dglnn
from sklearn.preprocessing import LabelEncoder, StandardScaler

# -------------------------
# Load and Prepare Dataset
# -------------------------
df = pd.read_csv("../all_circuits_features.csv")

label_encoder = LabelEncoder()
df["gate_label"] = label_encoder.fit_transform(df["gate_type"])

feature_columns = [
    "fan_in", "fan_out", "dist_to_output", "is_primary_input", "is_primary_output",
    "is_internal", "is_key_gate", "degree_centrality", "betweenness_centrality",
    "closeness_centrality", "clustering_coefficient", "avg_fan_in_neighbors", "avg_fan_out_neighbors"
]
df = df.dropna(subset=feature_columns)
df[feature_columns] = df[feature_columns].astype(float)
scaler = StandardScaler()
df[feature_columns] = scaler.fit_transform(df[feature_columns])

# -------------------------
# Build graph
# -------------------------
nodes = df["node"].tolist()
node_to_id = {node: i for i, node in enumerate(nodes)}
edges = []
for _, row in df.iterrows():
    node_id = node_to_id[row["node"]]
    potential_sources = df[df["fan_out"] > 0]["node"].tolist()
    k = int(row["fan_in"]) if not np.isnan(row["fan_in"]) else 0
    sources = potential_sources[:max(k, 0)]
    for src in sources:
        if src in node_to_id:
            edges.append((node_to_id[src], node_id))

src_nodes, dst_nodes = zip(*edges) if edges else ([], [])
graph = dgl.graph((torch.tensor(src_nodes), torch.tensor(dst_nodes)), num_nodes=len(nodes))
graph = dgl.add_self_loop(graph)
graph.ndata['features'] = torch.tensor(df[feature_columns].values, dtype=torch.float32)
graph.ndata['labels'] = torch.tensor(df["gate_label"].values, dtype=torch.long)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
graph = graph.to(device)
features = graph.ndata['features'].to(device)
labels = graph.ndata['labels'].to(device)

# -------------------------
# Class-incremental batches
# -------------------------
class_to_label = {c: i for i, c in enumerate(label_encoder.classes_)}
class_batches = [["and","not"],["nor"],["nand"],["input"],["output","or"],["xor"]]
all_labels_np = labels.cpu().numpy()
def indices_for_classes(class_names):
    class_ids = [class_to_label[c] for c in class_names]
    mask = np.isin(all_labels_np, class_ids)
    return np.where(mask)[0]
batch_indices = [indices_for_classes(b) for b in class_batches]

# -------------------------
# GCN model
# -------------------------
class GCN(nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats, dropout=0.5):
        super(GCN, self).__init__()
        self.conv1 = dglnn.GraphConv(in_feats, hidden_feats, allow_zero_in_degree=True)
        self.conv2 = dglnn.GraphConv(hidden_feats, out_feats, allow_zero_in_degree=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, g, inputs):
        x = self.conv1(g, inputs)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.conv2(g, x)
        return x

in_feats = features.shape[1]
out_feats = len(label_encoder.classes_)

# -------------------------
# Catastrophic Forgetting Metrics
# -------------------------
def weight_distance_metric(params_before, params_after):
    return sum(torch.norm(p_before - p_after, p=2).item()
               for p_before, p_after in zip(params_before, params_after))

def regularization_metric(params_before, params_after, fisher_matrix):
    return sum(torch.sum(F * (p_after - p_before) ** 2).item()
               for p_before, p_after, F in zip(params_before, params_after, fisher_matrix))

def replay_metric(acc_with_replay, acc_without_replay):
    return acc_with_replay - acc_without_replay

# -------------------------
# Training loop per mechanism
# -------------------------
def run_mechanism(mech, config):
    model = GCN(in_feats, config["hidden_feats"], out_feats, dropout=config["dropout"]).to(device)
    opt = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()

    params_before = [p.detach().clone() for p in model.parameters()]
    batch_accs = []

    for b, idx in enumerate(batch_indices):
        if b %10 == 0: print(f'Batch {b}.')
        train_idx_t = torch.tensor(idx, dtype=torch.long, device=device)
        for epoch in range(config["epochs_per_batch"]):
            model.train()
            logits = model(graph, features)
            loss = loss_fn(logits[train_idx_t], labels[train_idx_t])
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            logits = model(graph, features)
            preds = logits.argmax(dim=1)
            acc = (preds[train_idx_t] == labels[train_idx_t]).float().mean().item()
        batch_accs.append(acc)

    params_after = [p.detach().clone() for p in model.parameters()]
    wd = weight_distance_metric(params_before, params_after)
    fisher_matrix = [torch.ones_like(p) for p in params_before]
    reg = regularization_metric(params_before, params_after, fisher_matrix)
    replay_eff = None
    if mech in ["rehearsal","gen_replay"]:
        replay_eff = replay_metric(batch_accs[-1], batch_accs[0])

    return {"mechanism": mech,
            "batch_accs": batch_accs,
            "weight_distance": wd,
            "regularization": reg,
            "replay_effect": replay_eff}

# -------------------------
# Best hyperparameters per mechanism (from table)
# -------------------------
BEST_CONFIGS = {
    "naive":      {"hidden_feats":256,"dropout":0.0,"lr":0.001,"weight_decay":0.0,"epochs_per_batch":15},
    "rehearsal":  {"hidden_feats":128,"dropout":0.0,"lr":0.001,"weight_decay":0.0,"epochs_per_batch":27},
    "ewc":        {"hidden_feats":128,"dropout":0.0,"lr":0.001,"weight_decay":0.0,"epochs_per_batch":15},
    "si":         {"hidden_feats":256,"dropout":0.0,"lr":0.001,"weight_decay":0.0,"epochs_per_batch":9},
    "gen_replay": {"hidden_feats":256,"dropout":0.0,"lr":0.001,"weight_decay":0.0,"epochs_per_batch":20},
}

# -------------------------
# Run all mechanisms
# -------------------------
results = []
for mech, cfg in BEST_CONFIGS.items():
    print(f'Mechanism: {mech}')
    res = run_mechanism(mech, cfg)
    results.append(res)

# -------------------------
# Print results
# -------------------------
for r in results:
    print(f"\nMechanism: {r['mechanism']}")
    print(f"Batch Accuracies: {r['batch_accs']}")
    print(f"Weight Distance Metric: {r['weight_distance']:.4f}")
    print(f"Regularization Metric: {r['regularization']:.4f}")
    if r['replay_effect'] is not None:
        print(f"Replay Effectiveness Metric: {r['replay_effect']:.4f}")

Mechanism: naive
Batch 0.
Mechanism: rehearsal
Batch 0.
Mechanism: ewc
Batch 0.
Mechanism: si
Batch 0.
Mechanism: gen_replay
Batch 0.

Mechanism: naive
Batch Accuracies: [0.9417984485626221, 0.9286693930625916, 0.04016832262277603, 0.9928661584854126, 0.585669755935669, 1.0]
Weight Distance Metric: 4.1902
Regularization Metric: 7.2198

Mechanism: rehearsal
Batch Accuracies: [0.813043475151062, 0.9591906666755676, 0.7597551941871643, 0.9958035945892334, 0.5874499082565308, 1.0]
Weight Distance Metric: 4.9628
Regularization Metric: 10.1540
Replay Effectiveness Metric: 0.1870

Mechanism: ewc
Batch Accuracies: [0.8759288787841797, 0.7445130348205566, 0.0, 0.9995803833007812, 0.5874499082565308, 1.0]
Weight Distance Metric: 3.1251
Regularization Metric: 4.0286

Mechanism: si
Batch Accuracies: [0.550830066204071, 0.7030178308486938, 0.0, 0.9966428875923157, 0.0, 0.6967213153839111]
Weight Distance Metric: 2.7577
Regularization Metric: 3.1579

Mechanism: gen_replay
Batch Accuracies: [0.955138

#### Dynamic

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import dgl.nn as dglnn

# -------------------------
# Helpers (fresh per notebook)
# -------------------------
def _to_list(idx):
    if isinstance(idx, np.ndarray): return idx.tolist()
    if isinstance(idx, list): return idx
    return list(idx)

class RehearsalBuffer:
    def __init__(self, size=10000, num_nodes=None):
        self.size = size
        self.data_idx = []
        self.num_nodes = num_nodes
    def add(self, idx):
        idx_list = _to_list(idx)
        self.data_idx.extend(idx_list)
        if self.num_nodes is not None:
            self.data_idx = [i for i in self.data_idx if 0 <= i < self.num_nodes]
        self.data_idx = list(dict.fromkeys(self.data_idx))
        if len(self.data_idx) > self.size:
            self.data_idx = np.random.choice(self.data_idx, self.size, replace=False).tolist()
    def get(self):
        return self.data_idx

# -------------------------
# Dynamic GCN model (expandable output head)
# -------------------------
class DynamicGCN(nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats, dropout=0.5):
        super(DynamicGCN, self).__init__()
        self.conv1 = dglnn.GraphConv(in_feats, hidden_feats, allow_zero_in_degree=True)
        self.conv2 = dglnn.GraphConv(hidden_feats, hidden_feats, allow_zero_in_degree=True)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden_feats, out_feats)

    def forward(self, g, inputs):
        x = self.conv1(g, inputs)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.conv2(g, x)
        x = torch.relu(x)
        return self.out(x)

def expand_output_layer(model, new_out_feats, device):
    old_out = model.out
    old_out_feats = old_out.out_features
    if new_out_feats <= old_out_feats:
        return model, old_out_feats
    in_features = old_out.in_features
    new_out = nn.Linear(in_features, new_out_feats, bias=True).to(device)
    with torch.no_grad():
        nn.init.normal_(new_out.weight, mean=0.0, std=0.01)
        nn.init.zeros_(new_out.bias)
        new_out.weight[:old_out_feats, :] = old_out.weight
        new_out.bias[:old_out_feats] = old_out.bias
    model.out = new_out
    return model, old_out_feats

# -------------------------
# Catastrophic forgetting metrics (shape-aligned overlap)
# -------------------------
def _aligned_overlap(a, b):
    shape = tuple(min(x, y) for x, y in zip(a.shape, b.shape))
    slices = tuple(slice(0, s) for s in shape)
    return a[slices], b[slices]

def weight_distance_metric_aligned(params_before, params_after):
    total = 0.0
    for p0, p1 in zip(params_before, params_after):
        a, b = _aligned_overlap(p0, p1)
        total += torch.norm(a - b, p=2).item()
    return total

def regularization_metric_aligned(params_before, params_after, fisher_list):
    total = 0.0
    for p0, p1, F in zip(params_before, params_after, fisher_list):
        a, b = _aligned_overlap(p0, p1)
        Fa, _ = _aligned_overlap(F, F)
        total += torch.sum(Fa * (b - a) ** 2).item()
    return total

def replay_metric(acc_last, acc_first):
    return acc_last - acc_first

# -------------------------
# Dynamic training loop per mechanism (contiguous targets, expandable head)
# Relies on: graph, features, labels, class_batches, class_to_label, device defined in the static cell.
# -------------------------
in_feats = graph.ndata['features'].shape[1]  # recompute fresh

def run_dynamic_gcn(mech, config):
    # Initialize model to cover classes in the first batch
    first_ids = [class_to_label[c] for c in class_batches[0]]
    init_out_feats = max(first_ids) + 1
    model = DynamicGCN(in_feats, config["hidden_feats"], init_out_feats, dropout=config["dropout"]).to(device)
    opt = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()

    buffer = RehearsalBuffer(size=config.get("buffer_size", 0) or 10000,
                             num_nodes=graph.num_nodes()) if mech in ["rehearsal","gen_replay"] else None

    # Snapshot parameters before any training
    params_before = [p.detach().clone() for p in model.parameters()]
    batch_accs = []

    features = graph.ndata['features']
    labels = graph.ndata['labels']

    for b, idx in enumerate(batch_indices):
        # Classes seen so far and contiguous mapping
        seen_classes = [c for batch in class_batches[:b+1] for c in batch]
        seen_class_ids = [class_to_label[c] for c in seen_classes]
        target_out_feats = max(seen_class_ids) + 1

        # Expand output head if needed and reset optimizer
        if model.out.out_features < target_out_feats:
            model, _ = expand_output_layer(model, target_out_feats, device)
            opt = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

        # Training indices (+ replay if any)
        train_idx = _to_list(idx)
        if buffer is not None:
            train_idx += buffer.get()
            train_idx = list(dict.fromkeys(train_idx))
        train_idx_t = torch.tensor(train_idx, dtype=torch.long, device=device)

        # Map global labels -> contiguous [0..C_seen-1]
        id_to_seen = {cid: i for i, cid in enumerate(seen_class_ids)}
        max_cid = max(seen_class_ids)
        map_tensor = torch.full((max_cid + 1,), -1, dtype=torch.long, device=device)
        for cid, j in id_to_seen.items():
            map_tensor[cid] = j

        y_global = labels[train_idx_t]
        assert torch.all(map_tensor[y_global] != -1), f"Labels outside seen set at batch {b+1}"
        y_seen = map_tensor[y_global]

        seen_cols_t = torch.tensor(seen_class_ids, dtype=torch.long, device=device)

        # Train for K epochs
        for epoch in range(config["epochs_per_batch"]):
            model.train()
            logits_all = model(graph, features)
            logits_seen = logits_all[train_idx_t][:, seen_cols_t]
            loss = loss_fn(logits_seen, y_seen)
            opt.zero_grad(); loss.backward(); opt.step()

        # Update buffer
        if buffer is not None:
            buffer.add(idx)

        # Evaluate on training indices for seen slice
        with torch.no_grad():
            logits = model(graph, features)[:, seen_cols_t]
            preds = logits.argmax(dim=1)
            acc = (preds[train_idx_t] == y_seen).float().mean().item()
        batch_accs.append(acc)

    # Metrics with alignment to handle expansion
    params_after = [p.detach().clone() for p in model.parameters()]
    wd = weight_distance_metric_aligned(params_before, params_after)
    fisher_matrix = [torch.ones_like(p) for p in params_before]  # simple importance proxy
    reg = regularization_metric_aligned(params_before, params_after, fisher_matrix)
    replay_eff = replay_metric(batch_accs[-1], batch_accs[0]) if mech in ["rehearsal","gen_replay"] else None

    return {
        "mechanism": mech,
        "batch_accs": batch_accs,
        "weight_distance": wd,
        "regularization": reg,
        "replay_effect": replay_eff
    }

# -------------------------
# Best hyperparameters per mechanism (GCN Dynamic)
# -------------------------
BEST_CONFIGS = {
    "naive":      {"hidden_feats":64,  "dropout":0.5, "lr":0.0005, "weight_decay":0.0, "epochs_per_batch":18},
    "rehearsal":  {"hidden_feats":256, "dropout":0.5, "lr":0.0005, "weight_decay":0.0, "epochs_per_batch":60, "buffer_size":10000},
    "ewc":        {"hidden_feats":128, "dropout":0.5, "lr":0.0001, "weight_decay":0.0, "epochs_per_batch":90},
    "si":         {"hidden_feats":64,  "dropout":0.3, "lr":0.0005, "weight_decay":0.0, "epochs_per_batch":6},
    "gen_replay": {"hidden_feats":256, "dropout":0.5, "lr":0.0005, "weight_decay":0.0, "epochs_per_batch":90, "buffer_size":10000},
}

# -------------------------
# Run and print (second cell in the GCN dynamic notebook)
# -------------------------
results = []
for mech, cfg in BEST_CONFIGS.items():
    print(f"Running {mech}...")
    res = run_dynamic_gcn(mech, cfg)
    results.append(res)

for r in results:
    print(f"\nMechanism: {r['mechanism']}")
    print(f"Batch Accuracies: {r['batch_accs']}")
    print(f"Weight Distance Metric (aligned): {r['weight_distance']:.4f}")
    print(f"Regularization Metric (aligned): {r['regularization']:.4f}")
    if r['replay_effect'] is not None:
        print(f"Replay Effectiveness Metric: {r['replay_effect']:.4f}")


Running naive...
Running rehearsal...
Running ewc...
Running si...
Running gen_replay...

Mechanism: naive
Batch Accuracies: [0.7499802112579346, 0.9633058905601501, 0.11973986029624939, 0.9857322573661804, 0.09701824933290482, 0.9098360538482666]
Weight Distance Metric (aligned): 2.6424
Regularization Metric (aligned): 2.0570

Mechanism: rehearsal
Batch Accuracies: [0.9978063106536865, 0.9557138681411743, 0.87846839427948, 0.9120568633079529, 0.8780109286308289, 0.9035763740539551]
Weight Distance Metric (aligned): 20.7591
Regularization Metric (aligned): 203.8847
Replay Effectiveness Metric: -0.0942

Mechanism: ewc
Batch Accuracies: [0.9639130234718323, 1.0, 1.0, 0.9995803833007812, 0.41744548082351685, 1.0]
Weight Distance Metric (aligned): 4.3118
Regularization Metric (aligned): 6.4100

Mechanism: si
Batch Accuracies: [0.6877865791320801, 0.21296297013759613, 0.29112470149993896, 0.5413344502449036, 0.23097462952136993, 0.0901639312505722]
Weight Distance Metric (aligned): 0.9518
R